In [35]:
from agents.voice import workflow
from langgraph import graph
from langgraph.graph import START, END,StateGraph
from langchain_openai import AzureChatOpenAI
from  dotenv import load_dotenv
import os
from typing import TypedDict

In [36]:
load_dotenv()

AZURE_OPENAI_ENDPOINT = os.getenv("AZURE_OPENAI_ENDPOINT")
AZURE_OPENAI_API_KEY = os.getenv("AZURE_OPENAI_API_KEY")
AZURE_OPENAI_API_VERSION = os.getenv("AZURE_OPENAI_API_VERSION")
AZURE_OPENAI_DEPLOYMENT_NAME="gpt-4.1_1" #.env or this model : same :     working


In [40]:
llm_model = AzureChatOpenAI(azure_endpoint=AZURE_OPENAI_ENDPOINT,
api_version=AZURE_OPENAI_API_VERSION, azure_deployment=AZURE_OPENAI_DEPLOYMENT_NAME, temperature=0.2,)
answer1 = llm_model.invoke("ow far moon from earth?").content
answer1

'The average distance from the Earth to the Moon is about **384,400 kilometers** (approximately **238,855 miles**). This distance can vary slightly because the Moon’s orbit around the Earth is not a perfect circle, but rather an ellipse.'

In [41]:
class LLMChat(TypedDict):
    question: str
    answer : str


In [45]:
def llm_qa(state : LLMChat) -> LLMChat:
    question = state["question"]
    prompt = f'Answer following Question: {question}'
    answer = llm_model.invoke(prompt).content
    state["answer"] = answer
    return state

In [46]:
graph = StateGraph(LLMChat)

graph.add_node("LLN_QA", llm_qa)

graph.add_edge(START, "LLN_QA")
graph.add_edge("LLN_QA", END)

workflow = graph.compile()

In [48]:
intial_state = {"question" : "how far moon from earth?"}

final_state = workflow.invoke(intial_state)
# print(final_state)

print(final_state["answer"])


The average distance from the Moon to the Earth is about **384,400 kilometers** (approximately **238,855 miles**). This distance can vary slightly because the Moon’s orbit around the Earth is not a perfect circle.
